In [6]:
#0 Load Libraries and Configurations

import os
import wrds
import pandas as pd
import numpy as np

# ---------- User-configurable paths ----------
PATH_DATA_INTERMEDIATE = "/Users/nglei/Desktop/Academics/SMU/Modules/QF600 Asset Pricing/Project/Project Code/cz_data/intermediate"  # <-- change this
os.makedirs(PATH_DATA_INTERMEDIATE, exist_ok=True)

OUT_CSV     = os.path.join(PATH_DATA_INTERMEDIATE, "CRSPdistributions.csv")
OUT_PARQUET = os.path.join(PATH_DATA_INTERMEDIATE, "CRSPdistributions.parquet")

In [7]:
#1 Load CRSP Data

SQL = """
SELECT
    d.permno,
    d.divamt,
    d.distcd,
    d.facshr,
    d.rcrddt,
    d.exdt,
    d.paydt
FROM crsp.msedist AS d
WHERE (d.exdt   >= DATE '2000-01-01'
    OR d.rcrddt >= DATE '2000-01-01'
    OR d.paydt  >= DATE '2000-01-01');
"""

In [8]:
#2 CRSP Data Extraction From WRDS

db = wrds.Connection()
df = db.raw_sql(SQL, date_cols=["rcrddt", "exdt", "paydt"])

Enter your WRDS username [nglei]: nglei2025
Enter your password: ········


WRDS recommends setting up a .pgpass file.


Create .pgpass file now [y/n]?:  y


Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [10]:
#3 Data Cleaning

# -------- remove duplicates: by permno distcd paydt keep first --------
df = df.sort_values(["permno", "distcd", "paydt"], kind="mergesort")
df = df.drop_duplicates(subset=["permno", "distcd", "paydt"], keep="first")

# -------- distcd -> cd1..cd4 (as integers with NA support) --------
# Convert to string, left-pad to length 4, slice, and convert back to Int64
dist_str = (
    df["distcd"]
      .astype("Int64")              # handle NA safely
      .astype(str)
      .str.replace("<NA>", "", regex=False)
      .str.strip()
)
dist_str = dist_str.where(dist_str != "", other=np.nan)
dist_pad = dist_str.dropna().str.zfill(4)

df["cd1"] = pd.to_numeric(dist_pad.str[0], errors="coerce").astype("Int64")
df["cd2"] = pd.to_numeric(dist_pad.str[1], errors="coerce").astype("Int64")
df["cd3"] = pd.to_numeric(dist_pad.str[2], errors="coerce").astype("Int64")
df["cd4"] = pd.to_numeric(dist_pad.str[3], errors="coerce").astype("Int64")

# -------- save --------
df.to_csv(OUT_CSV, index=False)
df.to_parquet(OUT_PARQUET, index=False)

print("Saved:")
print(" -", OUT_CSV)
print(" -", OUT_PARQUET)
print(df.head())

Saved:
 - /Users/nglei/Desktop/Academics/SMU/Modules/QF600 Asset Pricing/Project/Project Code/cz_data/intermediate/CRSPdistributions.csv
 - /Users/nglei/Desktop/Academics/SMU/Modules/QF600 Asset Pricing/Project/Project Code/cz_data/intermediate/CRSPdistributions.parquet
    permno  divamt  distcd  facshr     rcrddt       exdt      paydt  cd1  cd2  \
23   10001   0.054    1222     0.0 2007-12-10 2007-12-06 2007-12-28    1    2   
24   10001   0.054    1222     0.0 2008-01-14 2008-01-10 2008-01-30    1    2   
26   10001   0.036    1222     0.0 2008-02-13 2008-02-11 2008-02-28    1    2   
27   10001   0.036    1222     0.0 2008-03-13 2008-03-11 2008-03-28    1    2   
28   10001   0.036    1222     0.0 2008-04-10 2008-04-08 2008-04-30    1    2   

    cd3  cd4  
23    2    2  
24    2    2  
26    2    2  
27    2    2  
28    2    2  
